In [ ]:
pip install neurodot_py

In [ ]:
pip install -r requirements.txt

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import neuro_dot as ndot
import plotly.io as pio
pio.renderers.default = 'notebook'  


# ── Load data ─────────────────────────────────────────────────────────────────
mat          = ndot.loadmat('hrf_DOT3.mat')
hrf          = mat['hrf'].ravel()

MNI, infoMNI = ndot.LoadVolumetricData('Segmented_MNI152nl_on_333_nifti', '', 'nii')

mesh_mat     = ndot.loadmat('MNI164k_big.mat')
MNIl         = mesh_mat['MNIl']
MNIr         = mesh_mat['MNIr']

T1, infoT1   = ndot.LoadVolumetricData('mni152nl_T1_on_333_nifti', '', 'nii')

data_mat  = ndot.loadmat('AC001_Reconstructed_Data_Sample.mat')
cortex_Hb = data_mat['cortex_Hb']
info      = data_mat['info']
dim       = info['tissue']['dim']
affine    = info['tissue']['affine']

# ── GLM ───────────────────────────────────────────────────────────────────────
params_AC = {
    'GVTD_censor': True,
    'GVTD_th':     0.001,
    'DoFilter':    True,
}

b, e, DM, EDM, keep, r_sqrd = ndot.GLM(
    np.squeeze(cortex_Hb[:, :, 0]),
    hrf,
    info,
    params_AC,
)

# ── Plot GLM Maps on the cortical surface──────────────────────────────────────
import plotly.offline as po
# ── Suppress po.plot throughout ───────────────────────────────────────────────
original_plot = po.plot
po.plot       = lambda *args, **kwargs: None

stimtypes = ['left', 'right', 'contrast']

# ── Compute beta volume for each stim type ─────────────────────────
def get_map(stim):
    if stim == 'left':
        b_contrast = b[:, 2] - b[:, 1]
    elif stim == 'right':
        b_contrast = b[:, 3] - b[:, 1]
    else:
        b_contrast = b[:, 2] - b[:, 3]
    bvol = ndot.GoodVox2vol(b_contrast.reshape(-1, 1), dim).squeeze()
    map_in_MNI_space = ndot.affine3d_img(bvol, dim, infoMNI, affine)
    return map_in_MNI_space, bvol

# ── Use first stim to get reference scene layout ──────────────────────────────
map_in_MNI_space, bvol = get_map(stimtypes[0])
nonzero_vals = bvol[bvol != 0]
ref_params = {
    'Scale': 0.8 * nonzero_vals.max() if nonzero_vals.size else 1.0,
    'Th':    {'P': 0, 'N': 0},
    'view':  'post',
    'ctx':   'std',
    'OL':    0,
}
_, _, ref_fig, _ = ndot.PlotInterpSurfMesh(
    map_in_MNI_space, MNIl, MNIr, infoMNI, ref_params
)
ref_scene = ref_fig.layout.scene.to_plotly_json()

# ── Build subplot figure ──────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=1,
    specs=[[{'type': 'scene'}]] * 3,
    subplot_titles=[f'Beta Map -- AC001/task: {s}' for s in stimtypes],
)

for j, stim in enumerate(stimtypes):

    map_in_MNI_space, bvol = get_map(stim)
    nonzero_vals            = bvol[bvol != 0]

    plot_params = {
        'Scale': 0.8 * nonzero_vals.max() if nonzero_vals.size else 1.0,
        'Th':    {'P': 0, 'N': 0},
        'view':  'post',
        'ctx':   'std',
        'OL':    0,
    }

    _, _, sub_fig, _ = ndot.PlotInterpSurfMesh(
        map_in_MNI_space, MNIl, MNIr, infoMNI, plot_params
    )

    for trace in sub_fig.data:
        fig.add_trace(trace, row=j+1, col=1 )
    # ── Copy scene layout from reference into each subplot ────────────────────
    scene_key = 'scene' if j == 0 else f'scene{j + 1}'
    fig.update_layout({scene_key: ref_scene})

# ── Global styling ────────────────────────────────────────────────────────────
fig.update_layout(
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white', size=8),
    title=dict(text='Beta Maps -- AC001', font=dict(color='white', size=16), x=0.5),
    height=1000,
    width=400,
    margin=dict(t=100),
)

fig.update_layout(
    scene=dict(  domain=dict(x=[0, 1], y=[0.70, 1.0])),
    scene2=dict( domain=dict(x=[0, 1], y=[0.36, 0.68])),
    scene3=dict( domain=dict(x=[0, 1], y=[0.02, 0.34])),
)
fig.update_layout(
    scene=dict(domain=dict(x=[0, 1], y=[0.70, 1.0])),
    scene2=dict(domain=dict(x=[0, 1], y=[0.36, 0.68])),
    scene3=dict(domain=dict(x=[0, 1], y=[0.02, 0.34])),
)

# Fix subplot title positions
for i, ann in enumerate(fig.layout.annotations):
    # i: 0 -> first subplot title, 1 -> second, 2 -> third
    if i == 0:
        ann.update(y=0.99)  # just above top scene
    elif i == 1:
        ann.update(y=0.67)  # just above middle scene
    elif i == 2:
        ann.update(y=0.33)  # just above bottom scene

# ── Restore and show ──────────────────────────────────────────────────────────
import plotly.io as pio
pio.renderers.default
po.plot = original_plot

fig.show()


In [ ]:
from scipy.io import savemat

# Save to mat file
outputFileName = 'GLM_output.mat'
savemat(outputFileName, {
    'b':         b,
    'e':         e,
    'DM':        DM,
    'EDM':       EDM,
    'keep':     keep,
    'r_sqrd': r_sqrd
})